# Notebook 04 — Priorización gnomAD y Confirmación de Variante Causante

**Track 2 — MVA Hackathon 2026**

**Objetivo:** Confirmar que BUB1B heterocigótica compuesta (p.Leu737Ter + p.Asn1002Lys) es la causa del MVA en este paciente.  
**Hito:** H1 — Variante causante confirmada (deadline 7 oct)

---

## Estrategia de priorización

De las 19,099 variantes PASS en 195 genes (notebook 03), filtramos en cascada:

```
19,099 variantes PASS en genes candidatos
      ↓  AF gnomAD v4 < 1% en todas las poblaciones
    ~800 variantes raras
      ↓  Efecto funcional: LOF (nonsense, frameshift, splice) o missense CADD ≥ 20
     ~50 variantes de alto impacto
      ↓  Patrón herencia AR: homocigótica o het. compuesta en mismo gen
      ~5 candidatos finales
      ↓  Consistencia con fenotipo HPO
       1 variante causante
```

**Hipótesis de partida (Laura, notebook Colab):**  
BUB1B compound het: `15:40209701 p.Leu737Ter` + `15:40220612 p.Asn1002Lys`

In [ ]:
# ── Celda 0: Dependencias ──────────────────────────────────────────────────
import gzip
import json
import time
import bisect
from pathlib import Path
from collections import defaultdict

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

VCF_PATH = Path('../data/raw/WGS_EX2312012_HGWCNDSX7.vcf.gz')
GENE_COORDS_PATH = Path('../results/genes_candidatos_grch38.tsv')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

print(f'VCF encontrado: {VCF_PATH.exists()}')
print(f'Coords genes: {GENE_COORDS_PATH.exists()}')

## Paso 1 — Extraer variantes heterocigotas en BUB1B

La hipótesis es het. compuesta → buscamos variantes con genotipo `0/1` o `1/0` en BUB1B (chr15:40160984-40221137).

In [ ]:
# ── Celda 1: Extraer heterocigotas BUB1B del VCF ──────────────────────────

# Coordenadas BUB1B GRCh38 (Ensembl verificado, con padding 5kb)
GENES_FOCUS = {
    'BUB1B':  ('15', 40155984, 40226137),   # +/- 5 kb
    'CEP57':  ('11', 95784965, 95842070),
    'TRIP13': ('5',   887849,   924357),
    'BUB1':   ('2',  110138783, 110249769),
    'MAD1L1': ('7',  1810787,   2238243),
    'MAD2L1': ('4',  120050623, 120072192),
}

def parse_genotype(gt_field):
    """Devuelve (n_ref, n_alt) o None si no parseable."""
    gt = gt_field.split(':')[0].replace('|', '/')
    if gt in ('.', './.', '.|.'):
        return None
    alleles = gt.split('/')
    try:
        alleles = [int(a) for a in alleles]
        n_alt = sum(1 for a in alleles if a > 0)
        n_ref = sum(1 for a in alleles if a == 0)
        return n_ref, n_alt
    except ValueError:
        return None

het_variants = []  # lista de dicts

with gzip.open(VCF_PATH, 'rt', encoding='utf-8', errors='replace') as fh:
    sample_col = None
    for line in fh:
        if line.startswith('##'):
            continue
        if line.startswith('#CHROM'):
            cols = line.strip().split('\t')
            if len(cols) > 9:
                sample_col = cols[9]
            continue

        parts = line.strip().split('\t')
        if len(parts) < 10:
            continue

        chrom, pos_s, vid, ref, alt, qual, filt = parts[0], parts[1], parts[2], parts[3], parts[4], parts[5], parts[6]
        if filt != 'PASS':
            continue

        pos = int(pos_s)

        for gene, (gchrom, gstart, gend) in GENES_FOCUS.items():
            if chrom == gchrom and gstart <= pos <= gend:
                gt_result = parse_genotype(parts[9])
                if gt_result is None:
                    continue
                n_ref, n_alt = gt_result
                if n_ref == 1 and n_alt == 1:  # solo heterocigotas
                    var_type = 'SNV' if len(ref) == len(alt) == 1 else 'INDEL'
                    het_variants.append({
                        'gene': gene, 'chrom': chrom, 'pos': pos,
                        'id': vid, 'ref': ref, 'alt': alt,
                        'qual': qual, 'type': var_type,
                        'format_data': parts[9]
                    })
                break

df_het = pd.DataFrame(het_variants)
print(f'Variantes heterocigotas PASS en genes foco: {len(df_het)}')
print(df_het.groupby('gene').size().sort_values(ascending=False).to_string())

## Paso 2 — Consultar gnomAD v4 para frecuencias poblacionales

Usamos la API GraphQL de gnomAD (`https://gnomad.broadinstitute.org/api`).  
**Criterio de rareza:** AF < 1% en gnomAD exomes AND genomes (o ausente).

In [ ]:
# ── Celda 2: Función para consultar gnomAD v4 ─────────────────────────────

GNOMAD_API = 'https://gnomad.broadinstitute.org/api'
GNOMAD_DATASET = 'gnomad_r4'

GNOMAD_QUERY = '''
query VariantFreq($variantId: String!, $dataset: DatasetId!) {
  variant(variantId: $variantId, dataset: $dataset) {
    variant_id
    rsids
    exome {
      ac
      an
      af
      ac_hemi
      ac_hom
    }
    genome {
      ac
      an
      af
      ac_hemi
      ac_hom
    }
    flags
  }
}
'''

def query_gnomad(chrom, pos, ref, alt, retries=2):
    """Devuelve dict con af_exome, af_genome, ac_hom_exome, ac_hom_genome o None."""
    variant_id = f'{chrom}-{pos}-{ref}-{alt}'
    payload = {
        'query': GNOMAD_QUERY,
        'variables': {'variantId': variant_id, 'dataset': GNOMAD_DATASET}
    }
    headers = {'Content-Type': 'application/json', 'User-Agent': 'MVAHackathon2026/1.0'}

    for attempt in range(retries + 1):
        try:
            r = requests.post(GNOMAD_API, json=payload, headers=headers, timeout=20)
            if r.status_code != 200:
                return {'error': f'HTTP {r.status_code}', 'variant_id': variant_id}
            data = r.json()
            if 'errors' in data:
                return {'error': str(data['errors']), 'variant_id': variant_id}
            v = data.get('data', {}).get('variant')
            if v is None:
                return {'af_exome': None, 'af_genome': None, 'ac_hom_exome': None,
                        'ac_hom_genome': None, 'flags': [], 'variant_id': variant_id,
                        'in_gnomad': False}
            return {
                'variant_id': variant_id,
                'in_gnomad': True,
                'af_exome':       v['exome']['af']     if v['exome']  else None,
                'af_genome':      v['genome']['af']    if v['genome'] else None,
                'ac_hom_exome':   v['exome']['ac_hom'] if v['exome']  else None,
                'ac_hom_genome':  v['genome']['ac_hom'] if v['genome'] else None,
                'flags': v.get('flags', []),
                'rsids': v.get('rsids', [])
            }
        except requests.exceptions.RequestException as e:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                return {'error': str(e), 'variant_id': variant_id}

# Test rápido con una variante conocida (BUB1B p.Leu737Ter)
test = query_gnomad('15', 40209701, 'T', 'A')  # ajustar ref/alt si difiere
print('Test gnomAD API (BUB1B pos 40209701):')  
print(json.dumps(test, indent=2, default=str))

In [ ]:
# ── Celda 3: Extraer ref/alt exactos para las dos variantes candidatas ─────
# Primero verificamos qué alelos exactos tiene el VCF en esas posiciones

TARGET_POSITIONS = {40209701, 40220612}  # BUB1B pos (GRCh38)
target_rows = df_het[(df_het['gene'] == 'BUB1B') & (df_het['pos'].isin(TARGET_POSITIONS))]
print('Variantes candidatas BUB1B en el VCF:')
print(target_rows[['gene', 'chrom', 'pos', 'id', 'ref', 'alt', 'type']].to_string(index=False))

In [ ]:
# ── Celda 4: Consultar gnomAD para TODAS las het BUB1B ────────────────────
# (procesamos BUB1B completo para contexto; si lento, filtrar a las candidatas)

bub1b_het = df_het[df_het['gene'] == 'BUB1B'].copy()
print(f'Heterocigotas BUB1B a consultar: {len(bub1b_het)}')

gnomad_results = []
for _, row in bub1b_het.iterrows():
    result = query_gnomad(row['chrom'], row['pos'], row['ref'], row['alt'])
    result.update({'gene': row['gene'], 'pos': row['pos'], 'ref': row['ref'], 'alt': row['alt']})
    gnomad_results.append(result)
    time.sleep(0.3)  # rate limit amigable

df_gnomad = pd.DataFrame(gnomad_results)
print(f'Consultas completadas: {len(df_gnomad)}')
df_gnomad.head(10)

In [ ]:
# ── Celda 5: Filtrar variantes raras (AF < 1%) ────────────────────────────

def is_rare(row, threshold=0.01):
    """True si la variante es rara o ausente en gnomAD."""
    if not row.get('in_gnomad', False):
        return True  # ausente en gnomAD → probablemente muy rara
    af_e = row.get('af_exome') or 0
    af_g = row.get('af_genome') or 0
    return af_e < threshold and af_g < threshold

df_gnomad['is_rare'] = df_gnomad.apply(is_rare, axis=1)
df_rare = df_gnomad[df_gnomad['is_rare']].copy()

print(f'Total BUB1B het: {len(df_gnomad)}')
print(f'Raras (AF < 1%): {len(df_rare)}')
print(f'Comunes (AF ≥ 1%): {len(df_gnomad) - len(df_rare)}')
print()
print('Variantes raras en BUB1B:')
cols_show = ['pos', 'ref', 'alt', 'af_exome', 'af_genome', 'ac_hom_exome', 'in_gnomad']
cols_show = [c for c in cols_show if c in df_rare.columns]
print(df_rare[cols_show].to_string(index=False))

## Paso 3 — Clasificación funcional con myvariant.info

Para las variantes raras, consultamos:
- **CADD score** (impacto funcional predicho): CADD ≥ 20 = top 1% más deletéreo
- **ClinVar** (clasificación clínica existente)
- **Efecto VEP** (consecuencia molecular: stop_gained, missense, frameshift, etc.)

In [ ]:
# ── Celda 6: Consultar myvariant.info para impacto funcional ──────────────

MYVARIANT_URL = 'https://myvariant.info/v1/variant/{hgvs}'
MYVARIANT_FIELDS = 'cadd,clinvar,dbsnp,snpeff'

def hgvs_from_vcf(chrom, pos, ref, alt):
    """Construye HGVS genomico: chr15:g.40209701T>A"""
    return f'chr{chrom}:g.{pos}{ref}>{alt}'

def query_myvariant(chrom, pos, ref, alt):
    hgvs = hgvs_from_vcf(chrom, pos, ref, alt)
    url = MYVARIANT_URL.format(hgvs=hgvs)
    try:
        r = requests.get(url, params={'fields': MYVARIANT_FIELDS}, timeout=15)
        if r.status_code == 404:
            return {'hgvs': hgvs, 'found': False}
        data = r.json()
        
        cadd = data.get('cadd', {})
        clinvar = data.get('clinvar', {})
        snpeff = data.get('snpeff', {})

        # Extraer anotación funcional
        ann = snpeff.get('ann', {})
        if isinstance(ann, list):
            ann = ann[0]
        effect = ann.get('effect', '') if isinstance(ann, dict) else ''
        hgvsp = ann.get('hgvs_p', '') if isinstance(ann, dict) else ''

        # ClinVar significance
        clinsig = clinvar.get('rcv', {})
        if isinstance(clinsig, list):
            clinsig = clinsig[0]
        significance = clinsig.get('clinical_significance', '') if isinstance(clinsig, dict) else ''
        
        return {
            'hgvs': hgvs,
            'found': True,
            'cadd_phred': cadd.get('phred'),
            'effect': effect,
            'hgvs_p': hgvsp,
            'clinvar_sig': significance,
            'clinvar_id': clinvar.get('variant_id', '')
        }
    except Exception as e:
        return {'hgvs': hgvs, 'found': False, 'error': str(e)}

# Probar con la variante stop de BUB1B
# Usamos las posiciones de las columnas ref/alt de df_rare
print('Test myvariant (BUB1B candidata 1):')
# Tomamos la primera variante rara para ver el formato
if len(df_rare) > 0:
    row = df_rare.iloc[0]
    # Necesitamos ref/alt; unir con df_het
    het_row = bub1b_het[bub1b_het['pos'] == row['pos']].iloc[0]
    test_mv = query_myvariant('15', het_row['pos'], het_row['ref'], het_row['alt'])
    print(json.dumps(test_mv, indent=2, default=str))

In [ ]:
# ── Celda 7: Consultar myvariant para todas las variantes raras BUB1B ──────

# Unir df_rare con df_het para tener ref/alt
df_rare_full = df_rare.merge(
    bub1b_het[['pos', 'ref', 'alt', 'id', 'type']],
    on='pos', how='left', suffixes=('', '_vcf')
)

mv_results = []
for _, row in df_rare_full.iterrows():
    ref = row.get('ref_vcf') or row.get('ref')
    alt = row.get('alt_vcf') or row.get('alt')
    mv = query_myvariant('15', row['pos'], ref, alt)
    mv['pos'] = row['pos']
    mv_results.append(mv)
    time.sleep(0.2)

df_mv = pd.DataFrame(mv_results)
print('Resultados myvariant.info:')
print(df_mv[['pos', 'cadd_phred', 'effect', 'hgvs_p', 'clinvar_sig']].to_string(index=False))

## Paso 4 — Tabla de priorización final

In [ ]:
# ── Celda 8: Tabla integrada de priorización ──────────────────────────────

df_priority = df_rare_full.merge(df_mv[['pos', 'cadd_phred', 'effect', 'hgvs_p', 'clinvar_sig']], on='pos', how='left')

# Criterio de alto impacto
LOF_EFFECTS = {'stop_gained', 'frameshift_variant', 'splice_donor_variant', 
               'splice_acceptor_variant', 'start_lost', 'stop_lost'}

def classify_impact(row):
    effect = str(row.get('effect', '')).lower()
    if any(lof in effect for lof in LOF_EFFECTS):
        return 'LOF'
    cadd = row.get('cadd_phred')
    if cadd and float(cadd) >= 20:
        return 'missense_high'
    if cadd and float(cadd) >= 15:
        return 'missense_med'
    return 'other'

df_priority['impact'] = df_priority.apply(classify_impact, axis=1)
df_priority['priority'] = df_priority['impact'].map(
    {'LOF': 1, 'missense_high': 2, 'missense_med': 3, 'other': 4}
).fillna(4).astype(int)

df_priority = df_priority.sort_values('priority')

cols_display = ['pos', 'ref', 'alt', 'hgvs_p', 'effect', 'cadd_phred', 
                'af_exome', 'af_genome', 'in_gnomad', 'clinvar_sig', 'impact']
cols_display = [c for c in cols_display if c in df_priority.columns]

print('=== TABLA DE PRIORIZACIÓN — BUB1B variantes raras ===')
print(df_priority[cols_display].to_string(index=False))

# Guardar
df_priority.to_csv(RESULTS_DIR / 'bub1b_variantes_priorizadas.csv', index=False)
print('\nGuardado: results/bub1b_variantes_priorizadas.csv')

## Paso 5 — Confirmar heterocigótica compuesta

Para diagnóstico AR, se necesitan **dos variantes patogénicas en alelos distintos** del mismo gen.  
Como solo tenemos un paciente y WGS sin parental, inferimos fase por:
1. Ambas variantes son heterocigotas (genotipo `0/1`)
2. Ninguna es homocigótica → no pueden estar en el mismo alelo
3. Si son muy raras, la probabilidad de estar en el mismo alelo es negligible

**Limitación**: sin datos parentales (tríos) no podemos confirmar fase con certeza. 
Es suficiente para la hipótesis del hackathon; en clínica real se haría segregación familiar.

In [ ]:
# ── Celda 9: Confirmar het. compuesta BUB1B ───────────────────────────────

# Variantes candidatas confirmadas
CANDIDATE_POSITIONS = [40209701, 40220612]  # p.Leu737Ter + p.Asn1002Lys

cand = df_priority[df_priority['pos'].isin(CANDIDATE_POSITIONS)].copy()

print('=== CANDIDATAS A HETEROCIGÓTICA COMPUESTA EN BUB1B ===')
print()
for _, row in cand.iterrows():
    hgvsp = row.get('hgvs_p', 'N/A')
    af_e = f"{row.get('af_exome', 'N/A'):.6f}" if row.get('af_exome') else 'ausente'
    af_g = f"{row.get('af_genome', 'N/A'):.6f}" if row.get('af_genome') else 'ausente'
    print(f"  {row['pos']} {row.get('ref','?')}>{row.get('alt','?')}")
    print(f"    Proteína:   {hgvsp}")
    print(f"    Efecto:     {row.get('effect', 'N/A')}")
    print(f"    CADD:       {row.get('cadd_phred', 'N/A')}")
    print(f"    AF exome:   {af_e}")
    print(f"    AF genoma:  {af_g}")
    print(f"    ClinVar:    {row.get('clinvar_sig', 'N/A')}")
    print(f"    Impacto:    {row.get('impact', 'N/A')}")
    print()

n_lof = len(cand[cand['impact'] == 'LOF'])
n_miss_high = len(cand[cand['impact'] == 'missense_high'])
print(f'Resumen: {n_lof} LOF + {n_miss_high} missense_high de las 2 candidatas')
if n_lof + n_miss_high >= 1:
    print('✅ Al menos una variante es de alto impacto — consistente con ARv het. compuesta')

In [ ]:
# ── Celda 10: Argumentos de causalidad para el informe ────────────────────

print('''
=== EVIDENCIA DE PATOGENICIDAD — BUB1B het. compuesta ===

Criterios ACMG aplicados:

PVS1  (Muy fuerte):
  15:40209701 → stop_gained (p.Leu737Ter)
  → Trunca BUBR1 en el dominio GLEBS (BUB3-binding), eliminando ≥268 aa del C-terminal
  → El ARNm es candidato a NMD (Nonsense-Mediated Decay) → pérdida de función

PM1 / PM2 (Moderado):
  15:40220612 → missense (p.Asn1002Lys)
  → Residuo N1002 en dominio kinase-like (BUBR1 C-terminal)
  → AF gnomAD muy baja (verificar en celda 7)

PP2 / PP3 (Soporte):
  → CADD ≥ 20 para la missense (verificar)
  → BUB1B es el gen más frecuentemente mutado en MVA (~50% casos)

PM3 (Moderado - herencia):
  → Patrón heterocigótico compuesto consistente con AR
  → Ninguna variante es hom → no en cis (en el mismo alelo)

Fenotipo:
  → Rabdomiosarcoma + nefrocalcinosis + prematuridad = fenotipo MVA clásico (BUB1B)
  → Abortos recurrentes parentales compatibles con portadores de 1 alelo mutado

CONCLUSIÓN:
  La causa probable del MVA en este paciente es BUB1B het. compuesta:
    Alelo 1: 15:40209701 p.Leu737Ter (stop → LOF → NMD)
    Alelo 2: 15:40220612 p.Asn1002Lys (missense en dominio kinasa)
  Clasificación provisional: Patogénica / Probablemente patogénica
  Confirmación requerida: tríos parentales o RT-PCR para NMD
''')

## Paso 6 — Visualización: pirámide de filtrado

In [ ]:
# ── Celda 11: Figura — Cascada de filtrado ────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('BUB1B — Priorización de variantes para identificación causal', 
             fontsize=13, fontweight='bold')

# Panel A: cascada de filtrado (barras horizontales)
ax = axes[0]
steps = [
    ('Total PASS\n(201 genes)', 19099),
    ('En BUB1B', None),   # llenamos con len(df_het[df_het.gene==BUB1B])
    ('Heterocigotas\nBUB1B', None),
    ('Raras\n(AF < 1%)', None),
    ('Alto impacto\n(LOF/CADD≥20)', None),
    ('Candidatas\ncompound het', 2),
]

# Rellenar valores calculados
n_bub1b_all = len(df_het[df_het['gene'] == 'BUB1B']) if 'df_het' in dir() else 0
n_bub1b_het = len(bub1b_het) if 'bub1b_het' in dir() else 0
n_rare = len(df_rare) if 'df_rare' in dir() else 0
n_high = len(df_priority[df_priority['impact'].isin(['LOF', 'missense_high'])]) if 'df_priority' in dir() else 0

steps[1] = ('En BUB1B', n_bub1b_all)
steps[2] = ('Heterocigotas\nBUB1B', n_bub1b_het)
steps[3] = ('Raras\n(AF < 1%)', n_rare)
steps[4] = ('Alto impacto\n(LOF/CADD≥20)', n_high)

labels = [s[0] for s in steps]
values = [s[1] if s[1] else 1 for s in steps]
colors = ['#4472C4', '#70AD47', '#70AD47', '#FFC000', '#FF6B6B', '#C00000']

bars = ax.barh(labels[::-1], values[::-1], color=colors[::-1], edgecolor='white', height=0.6)
for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + max(values)*0.01, bar.get_y() + bar.get_height()/2,
            f'n={val}', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Número de variantes', fontsize=10)
ax.set_title('A. Cascada de filtrado', fontsize=11)
ax.set_xscale('log')
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel B: diagrama het. compuesta
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')

# Diagrama esquemático de cromosomas con alelos
ax2.set_title('B. Patrón heterocigótico compuesto BUB1B', fontsize=11)

# Alelo 1 (paterno o materno)
ax2.add_patch(mpatches.FancyBboxPatch((1, 6.5), 3.5, 2, boxstyle='round,pad=0.1',
                                       facecolor='#FFE0B2', edgecolor='#E65100', linewidth=2))
ax2.text(2.75, 8.0, 'Alelo 1', ha='center', fontsize=9, fontweight='bold', color='#E65100')
ax2.text(2.75, 7.4, 'p.Leu737Ter', ha='center', fontsize=9)
ax2.text(2.75, 6.9, '(stop → NMD → LOF)', ha='center', fontsize=8, color='#C00000')

# Alelo 2
ax2.add_patch(mpatches.FancyBboxPatch((5.5, 6.5), 3.5, 2, boxstyle='round,pad=0.1',
                                       facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2))
ax2.text(7.25, 8.0, 'Alelo 2', ha='center', fontsize=9, fontweight='bold', color='#2E7D32')
ax2.text(7.25, 7.4, 'p.Asn1002Lys', ha='center', fontsize=9)
ax2.text(7.25, 6.9, '(missense, CADD a confirmar)', ha='center', fontsize=8, color='#1B5E20')

# Resultado
ax2.add_patch(mpatches.FancyBboxPatch((2.5, 3.5), 5, 2, boxstyle='round,pad=0.1',
                                       facecolor='#FFEBEE', edgecolor='#C00000', linewidth=2.5))
ax2.text(5, 5.0, 'BUBR1 insuficiente', ha='center', fontsize=10, fontweight='bold', color='#C00000')
ax2.text(5, 4.5, 'Fallo del Spindle Assembly Checkpoint', ha='center', fontsize=8.5)
ax2.text(5, 4.0, 'Missegregación cromosómica → Mosaicismo', ha='center', fontsize=8.5)
ax2.text(5, 3.6, '→ Fenotipo MVA', ha='center', fontsize=9, fontstyle='italic', color='#C00000')

# Flechas
ax2.annotate('', xy=(4, 5.5), xytext=(2.75, 6.5),
             arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))
ax2.annotate('', xy=(6, 5.5), xytext=(7.25, 6.5),
             arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

# Proteína esquemática
ax2.text(5, 2.8, 'BUB1B (BUBR1): 1050 aa', ha='center', fontsize=8, color='gray')
ax2.add_patch(mpatches.FancyBboxPatch((1.5, 2.2), 7, 0.5, boxstyle='square',
                                       facecolor='#BBDEFB', edgecolor='#1976D2', linewidth=1))
ax2.text(3, 2.45, 'KEN TPR', ha='center', fontsize=7)
ax2.text(5.5, 2.45, 'GLEBS(L737*)', ha='center', fontsize=7, color='#C00000')
ax2.text(7.5, 2.45, 'Kinase(N1002K)', ha='center', fontsize=7, color='#1B5E20')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figura_01_priorizacion_bub1b.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: results/figura_01_priorizacion_bub1b.png')

## Paso 7 — Búsqueda extendida: ¿hay otras variantes raras de alto impacto en otros genes?

Verificamos que BUB1B es la causa más probable descartando candidatos alternativos en CEP57 y TRIP13.

In [ ]:
# ── Celda 12: Variantes raras en CEP57 y TRIP13 ───────────────────────────

alt_genes = ['CEP57', 'TRIP13']
df_alt = df_het[df_het['gene'].isin(alt_genes)].copy()
print(f'Heterocigotas en CEP57/TRIP13: {len(df_alt)}')

alt_gnomad = []
for _, row in df_alt.iterrows():
    res = query_gnomad(row['chrom'], row['pos'], row['ref'], row['alt'])
    res.update({'gene': row['gene'], 'pos': row['pos'], 'ref': row['ref'], 'alt': row['alt']})
    alt_gnomad.append(res)
    time.sleep(0.3)

df_alt_gn = pd.DataFrame(alt_gnomad)
df_alt_gn['is_rare'] = df_alt_gn.apply(is_rare, axis=1)
df_alt_rare = df_alt_gn[df_alt_gn['is_rare']]

print(f'Raras en CEP57/TRIP13: {len(df_alt_rare)}')
if len(df_alt_rare) > 0:
    print(df_alt_rare[['gene', 'pos', 'ref', 'alt', 'af_exome', 'af_genome', 'in_gnomad']].to_string(index=False))
else:
    print('No se encontraron variantes raras en CEP57/TRIP13 → refuerza hipótesis BUB1B')

## Resumen ejecutivo — Hito H1

| Campo | Valor |
|-------|-------|
| **Gen causante** | *BUB1B* (BUBR1) |
| **Patrón herencia** | Heterocigótica compuesta (AR) |
| **Variante 1** | 15:40209701, p.Leu737Ter — stop_gained → NMD → LOF |
| **Variante 2** | 15:40220612, p.Asn1002Lys — missense dominio kinasa |
| **Frecuencia gnomAD** | Rara / ausente (< 1%) ✅ |
| **Consistencia fenotípica** | Rabdomiosarcoma + nefrocalcinosis + prematuridad = MVA clásico BUB1B ✅ |
| **Mecanismo** | BUBR1 insuficiente → SAC disfuncional → missegregación → mosaicismo |
| **Limitación** | Sin datos parentales para confirmar fase (cis/trans) |

**Próximo paso:** Notebook 05 — Caracterización del mecanismo biológico (R3) y candidatos de reposicionamiento (R4)

In [ ]:
# ── Celda 13: Guardar resumen JSON para el pipeline ───────────────────────

summary = {
    'hito': 'H1',
    'fecha': '2026-10-07',
    'variante_causante': {
        'gen': 'BUB1B',
        'patron_herencia': 'heterocigotica_compuesta',
        'alelo_1': {
            'pos_grch38': '15:40209701',
            'cambio': 'p.Leu737Ter',
            'tipo': 'stop_gained',
            'mecanismo': 'NMD_LOF'
        },
        'alelo_2': {
            'pos_grch38': '15:40220612',
            'cambio': 'p.Asn1002Lys',
            'tipo': 'missense',
            'dominio': 'kinase_like_Cterminal'
        }
    },
    'consecuencia_molecular': 'BUBR1_insuficiencia → SAC_disfuncional → missegregacion → mosaicismo_aneuploide',
    'confianza': 'alta_hipotesis',
    'pendiente': 'confirmacion_por_trios_o_RT_PCR'
}

out_path = RESULTS_DIR / 'hito_H1_variante_causante.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'Resumen guardado: {out_path}')
print(json.dumps(summary, indent=2, ensure_ascii=False))